# Molecular docking lab: AutoDock Vina

You will dock a known drug into its protein target, check that your setup can reproduce the crystal structure, then dock your own compound. Run each cell in order with **Shift+Enter**. Edit only the cells marked **Your settings**.

The matching step-by-step guide, with the box builder and methods-paragraph generator, is on the course page linked from the repository README.

## Setup

In [ ]:
import os, sys, importlib.metadata as md
from pathlib import Path

SCRIPTS = Path("..").resolve() / "scripts"      # helper functions live here
sys.path.insert(0, str(SCRIPTS))
import dock_utils as du
import pandas as pd
import py3Dmol

WORK = Path("work"); WORK.mkdir(exist_ok=True)   # all files are written here
os.chdir(WORK)
print("Vina", md.version("vina"), "| Meeko", md.version("meeko"))

## Your settings

Run these once with the defaults (HIV-1 protease with indinavir) for the validation. For your own proposal, change them in Part 5.

In [ ]:
PDB_ID = "1HSG"             # PDB entry of the target
LIGAND_RESNAME = "MK1"      # three-letter name of the co-crystal ligand
EXHAUSTIVENESS = 32         # search effort. Use 8 for a quick test, 32 for real runs
PADDING = 8.0               # Å added around the ligand to make the search box

# Offline classroom? Put your own files in the work/ folder and set these:
LOCAL_COMPLEX = None        # e.g. "1HSG.pdb"
LOCAL_LIGAND_SDF = None     # e.g. "MK1_ideal.sdf"

## Part 1: Inspect the structure

Download the entry and look at it. The ligand is shown in sticks and the protein as a cartoon.

In [ ]:
pdb_file = Path(LOCAL_COMPLEX) if LOCAL_COMPLEX else du.fetch_pdb(PDB_ID)

view = py3Dmol.view(width=720, height=460)
view.addModel(Path(pdb_file).read_text(), "pdb")
view.setStyle({"cartoon": {"color": "spectrum", "opacity": 0.7}})
view.addStyle({"resn": LIGAND_RESNAME}, {"stick": {"colorscheme": "yellowCarbon", "radius": 0.25}})
view.zoomTo({"resn": LIGAND_RESNAME})
view.show()

**Answer in your notes:** How many chains are there? Which residues are closest to the ligand? Where are the waters and are any near the ligand?

## Part 2: Prepare the receptor and the ligand

Split the complex into protein and crystal ligand, then convert both to PDBQT. Read the list of dropped heteroatoms: waters are expected, but a metal or cofactor near the site should be kept.

In [ ]:
receptor_pdb, crystal_pdb = du.split_complex(pdb_file, LIGAND_RESNAME)
receptor_pdbqt = du.prepare_receptor(receptor_pdb, "receptor")

ideal_sdf = Path(LOCAL_LIGAND_SDF) if LOCAL_LIGAND_SDF else du.fetch_ligand_sdf(LIGAND_RESNAME)
control_pdbqt = du.prepare_ligand(ideal_sdf, "control.pdbqt")

## Part 3: Define the search box

The box is centered on the crystal ligand and padded on every side. Vina only searches inside it.

In [ ]:
center, size = du.box_from_ligand(crystal_pdb, padding=PADDING)

view = py3Dmol.view(width=720, height=460)
view.addModel(Path(receptor_pdb).read_text(), "pdb")
view.setStyle({"cartoon": {"color": "white", "opacity": 0.6}})
view.addModel(Path(crystal_pdb).read_text(), "pdb")
view.setStyle({"model": 1}, {"stick": {"colorscheme": "yellowCarbon"}})
view.addBox({"center": dict(x=center[0], y=center[1], z=center[2]),
             "dimensions": dict(w=size[0], h=size[1], d=size[2]),
             "color": "teal", "opacity": 0.25})
view.zoomTo({"model": 1})
view.show()

**Check:** does the ligand sit comfortably inside the box, with room around it?

## Part 4: Validate by redocking

Dock the crystal ligand back into its own pocket. If the top pose lands within **2 Å** RMSD of the crystal pose, the setup is trustworthy enough to use on new compounds.

This takes a few minutes with `EXHAUSTIVENESS = 32`.

In [ ]:
control_out = Path("control_out.pdbqt")
control_affinities = du.dock(receptor_pdbqt, control_pdbqt, center, size,
                             control_out, exhaustiveness=EXHAUSTIVENESS)
control_sdf = du.poses_to_sdf(control_out, "control_poses.sdf")
control_rmsd = du.rmsd_to_crystal(control_sdf, crystal_pdb, template_sdf=ideal_sdf)

verdict = "PASS" if control_rmsd[0] <= 2.0 else "FAIL"
print(f"\nTop pose: {control_affinities[0]} kcal/mol, RMSD {control_rmsd[0]} Å -> {verdict}")

In [ ]:
view = py3Dmol.view(width=720, height=460)
view.addModel(Path(receptor_pdb).read_text(), "pdb")
view.setStyle({"cartoon": {"color": "white", "opacity": 0.5}})
view.addModel(Path(crystal_pdb).read_text(), "pdb")
view.setStyle({"model": 1}, {"stick": {"color": "limegreen"}})          # crystal
view.addModel(Path(control_sdf).read_text().split("$$$$")[0], "sdf")
view.setStyle({"model": 2}, {"stick": {"color": "magenta"}})            # top pose
view.zoomTo({"model": 2})
view.show()

Green is the crystal ligand and magenta is the top docked pose. Describe which groups overlap and which differ. If the result is **FAIL**, check the box, the protonation, and the exhaustiveness, and report the outcome honestly.

**Try it:** set `EXHAUSTIVENESS = 2` and run Part 4 again. What happens to the RMSD, and why?

## Part 5: Dock your own compound

Give your compound as a SMILES string, written in the protonation state you want at pH 7.4 (for example `C(=O)[O-]` for a carboxylate). The same receptor and box are reused so the scores are comparable with the control.

In [ ]:
MY_NAME = "my_compound"
MY_SMILES = "CC(=O)Oc1ccccc1C(=O)O"     # replace with your compound (this is aspirin)

my_sdf = du.ligand_from_smiles(MY_SMILES, f"{MY_NAME}.sdf")
my_pdbqt = du.prepare_ligand(my_sdf, f"{MY_NAME}.pdbqt")
my_out = Path(f"{MY_NAME}_out.pdbqt")
my_affinities = du.dock(receptor_pdbqt, my_pdbqt, center, size, my_out,
                        exhaustiveness=EXHAUSTIVENESS)
my_poses = du.poses_to_sdf(my_out, f"{MY_NAME}_poses.sdf")

In [ ]:
near, polar = du.contacts(receptor_pdb, my_poses, pose_index=0)
print("Residues within 4 Å of the top pose:")
print(", ".join(near))
print("\nCandidate polar contacts (N/O pairs shorter than 3.3 Å):")
for res, atom, lig_el, d in polar:
    print(f"  {res:10s} {atom:5s} ... ligand {lig_el}   {d} Å")

In [ ]:
view = py3Dmol.view(width=720, height=460)
view.addModel(Path(receptor_pdb).read_text(), "pdb")
view.setStyle({"cartoon": {"color": "white", "opacity": 0.5}})
residues = sorted({int(r.split(":")[0][3:]) for r in near})
view.addStyle({"resi": residues}, {"stick": {"colorscheme": "cyanCarbon", "radius": 0.12}})
view.addModel(Path(my_poses).read_text().split("$$$$")[0], "sdf")
view.setStyle({"model": 1}, {"stick": {"color": "magenta"}})
view.zoomTo({"model": 1})
view.show()

## Results table and methods paragraph

Compare your compound with the control. Scores near each other (within about 1 kcal/mol) should not be treated as different.

In [ ]:
results = pd.DataFrame([
    {"Compound": f"{LIGAND_RESNAME} (control)", "Top affinity (kcal/mol)": control_affinities[0],
     "RMSD to crystal (Å)": control_rmsd[0]},
    {"Compound": MY_NAME, "Top affinity (kcal/mol)": my_affinities[0], "RMSD to crystal (Å)": None},
])
results.to_csv("results.csv", index=False)
results

In [ ]:
print(f"""Methods (edit before you use it):
Compounds were docked into {PDB_ID} using AutoDock Vina {md.version("vina")}. The receptor was prepared
with Meeko {md.version("meeko")} after removing waters and heteroatoms, and ligands were converted to PDBQT
with Meeko. The search box ({size[0]} x {size[1]} x {size[2]} Å) was centered on the co-crystal ligand
{LIGAND_RESNAME} at ({center[0]}, {center[1]}, {center[2]}), with exhaustiveness {EXHAUSTIVENESS}.
Redocking of {LIGAND_RESNAME} reproduced the crystal pose with an RMSD of {control_rmsd[0]} Å.
{MY_NAME} was predicted to bind with an affinity of {my_affinities[0]} kcal/mol.
These scores are used to prioritize hypotheses for testing and are not measured affinities.""")

Files from this session are in the `work/` folder. In Codespaces, right-click a file in the Explorer and choose **Download**. On Binder, copy anything you need before the session closes, because it is deleted when it times out.